# Step 3: Pretrain Transformer Generator

Transformer pretraining on filtered ChEMBL corpus (next-token prediction).

In [ ]:
from molrl.dataloader import create_dataloader
from molrl.models import AutoregressiveTransformer
from flax import nnx
import jax
from jax import numpy as jnp
import optax
import orbax.checkpoint as ocp
from pathlib import Path
from molrl.utils import generate_molecules_temperature
from molrl.eval import smiles_validity, smiles_uniqueness, smiles_novelty
from molrl.training import train_step, val_step
from tqdm.asyncio import tqdm
import pandas as pd 

In [2]:
hypers = {
    "dataset": "chembl",
    "batch_size": 256,
    "num_workers": 4,
    "max_num_updates": 100_000,
    "n_eval_batches": 32,
    "eval_every": 2500,
    "early_stop_patience_evals": 5,
    "learning_rate": 3e-4,
    "warmup_steps": 2000,
    "model_save_path": "checkpoints/pretrained_transformer",
    "vocab_size": 36,
    "max_seq_len": 102,
    "emb_dim": 256,
    "num_layers": 6,
    "num_heads": 8,
    "mlp_dim": 1024,
    "dropout_rate": 0.1
}

# Init everything

In [3]:
# init the dataloaders
train_loader = create_dataloader("../step_1_data_preparation/chembl_train_aug.h5",
                                      batch_size=hypers["batch_size"],
                                      shuffle=True,
                                      to_device=True,
                                      )

val_loader = create_dataloader("../step_1_data_preparation/chembl_val_aug.h5",
                                      batch_size=hypers["batch_size"],
                                      shuffle=True,
                                      to_device=True,
                                      )
# init the model
model = AutoregressiveTransformer(vocab_size=hypers["vocab_size"], 
                                  max_seq_len=hypers["max_seq_len"], 
                                  emb_dim=hypers["emb_dim"], 
                                  num_layers=hypers["num_layers"], 
                                  num_heads=hypers["num_heads"], 
                                  mlp_dim=hypers["mlp_dim"], 
                                  dropout_rate=hypers["dropout_rate"])

# init the optimizer with linear warmup + cosine decay
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=hypers["learning_rate"],
    warmup_steps=hypers["warmup_steps"],
    decay_steps=hypers["max_num_updates"],
    end_value=hypers["learning_rate"] * 0.1,
)
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=schedule), wrt=nnx.Param)

In [ ]:
n_updates = 0
train_losses = []
best_val_loss = float("inf")
no_improve_evals = 0
stop_training = False

ckpt_dir = (Path.cwd() / hypers["model_save_path"]).resolve()
ckpt_dir.parent.mkdir(parents=True, exist_ok=True)
checkpointer = ocp.PyTreeCheckpointer()

while n_updates < hypers["max_num_updates"] and not stop_training:
    for batch in train_loader:
        loss = train_step(model, optimizer, batch, pad_token_id=0)
        train_losses.append(float(loss))
        n_updates += 1

        if n_updates % hypers["eval_every"] == 0:
            print(f"Evaluating at step {n_updates}...")
            current_train_loss = sum(train_losses) / len(train_losses)
            train_losses = []  # reset for next interval
            val_losses = []
            
            for i, val_batch in enumerate(val_loader):
                if i < hypers["n_eval_batches"]:
                    val_losses.append(float(val_step(model, val_batch, pad_token_id=0)))
                else:
                    break
            val_loss = sum(val_losses) / len(val_losses)

            designs = generate_molecules_temperature(model, max_seq_len=model.max_seq_len, t=1.0, n=1000)
            validity = smiles_validity(designs)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                no_improve_evals = 0
                checkpointer.save(str(ckpt_dir), nnx.state(model), force=True)
                print(f"New best val loss: {best_val_loss:.4f}. Saved checkpoint to {ckpt_dir}")
            else:
                no_improve_evals += 1
                print(f"No val improvement for {no_improve_evals}/{hypers['early_stop_patience_evals']} eval checks")

            print(f"Step {n_updates}, Train loss: {current_train_loss:.4f}, Val loss: {val_loss:.4f}, Validity: {validity:.4f}")

            if no_improve_evals >= hypers["early_stop_patience_evals"]:
                print("Early stopping triggered.")
                stop_training = True
                break

        if n_updates >= hypers["max_num_updates"]:
            break

if best_val_loss == float("inf"):
    checkpointer.save(str(ckpt_dir), nnx.state(model), force=True)
    print(f"Training ended before eval. Saved fallback checkpoint to {ckpt_dir}")
else:
    print(f"Training finished. Best val loss: {best_val_loss:.4f}")

Evaluating at step 2500...
New best val loss: 0.9642. Saved checkpoint to /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_2_pretrain_generator/checkpoints/pretrained_transformer
Step 2500, Train loss: 1.3508, Val loss: 0.9642, Validity: 0.3650
Evaluating at step 5000...
New best val loss: 0.8881. Saved checkpoint to /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_2_pretrain_generator/checkpoints/pretrained_transformer
Step 5000, Train loss: 0.9565, Val loss: 0.8881, Validity: 0.7290
Evaluating at step 7500...
New best val loss: 0.8551. Saved checkpoint to /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_2_pretrain_generator/checkpoints/pretrained_transformer
Step 7500, Train loss: 0.9006, Val loss: 0.8551, Validity: 0.8070
Evaluating at step 10000...
New best val loss: 0.8397. Saved checkpoint to /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_2_pretrain_generator/checkpoints/pretrained_transformer
Step 1000

In [ ]:
# ckpt_dir = (Path.cwd() / hypers["model_save_path"]).resolve()
# ckpt_dir.parent.mkdir(parents=True, exist_ok=True)
# checkpointer = ocp.PyTreeCheckpointer()
# checkpointer.save(str(ckpt_dir), nnx.state(model), force=True)
# print(f"Saved checkpoint to {ckpt_dir}")

In [22]:
# Load the checkpoint into a fresh model instance.
from molrl.training import val_step

ckpt_dir = (Path.cwd() / hypers["model_save_path"]).resolve()
if not ckpt_dir.exists():
    raise FileNotFoundError(f"Checkpoint not found at {ckpt_dir}. Run the training loop cell first.")

checkpointer = ocp.PyTreeCheckpointer()

model = AutoregressiveTransformer(
    vocab_size=hypers["vocab_size"],
    max_seq_len=hypers["max_seq_len"],
    emb_dim=hypers["emb_dim"],
    num_layers=hypers["num_layers"],
    num_heads=hypers["num_heads"],
    mlp_dim=hypers["mlp_dim"],
    dropout_rate=hypers["dropout_rate"],
)

# Restore using a template state so list/dict key types match NNX internals.
restored_state = checkpointer.restore(str(ckpt_dir), item=nnx.state(model))
nnx.update(model, restored_state)

# Quick sanity check: restored model should run on a validation batch.
test_batch = next(iter(val_loader))
restored_val_loss = float(val_step(model, test_batch, pad_token_id=0))
print(f"Restored model validation loss: {restored_val_loss:.4f}")

/Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/.venv/lib/python3.13/site-packages/orbax/checkpoint/_src/serialization/jax_array_handlers.py:712: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Restored model validation loss: 0.7360


In [24]:
test_loader = create_dataloader("../step_1_data_preparation/chembl_test_aug.h5",
                                      batch_size=hypers["batch_size"],
                                      shuffle=False,
                                      to_device=True,
                                      )

test_losses = []
for test_batch in test_loader:
    test_loss = float(val_step(model, test_batch, pad_token_id=0))
    test_losses.append(test_loss)

test_loss = sum(test_losses) / len(test_losses)
print(f"Average test loss: {test_loss:.4f}")

Average test loss: 0.7460


In [25]:
designs = []
total_designs = 10_000
for _ in tqdm(range(total_designs // 1000)):  # Generate 100k molecules in total
    designs_batch = generate_molecules_temperature(model, max_seq_len=model.max_seq_len, t=1.0, n=1000)
    designs.extend(designs_batch)


100%|██████████| 10/10 [09:38<00:00, 57.87s/it]


In [26]:
train_smiles = pd.read_csv("../step_1_data_preparation/chembl_train.csv")["standardized_smiles"].tolist()

validity = smiles_validity(designs)
uniqueness = smiles_uniqueness(designs)
novelty = smiles_novelty(designs, training_smiles=train_smiles)

print(f"Validity: {validity:.4f}, Uniqueness: {uniqueness:.4f}, Novelty: {novelty:.4f}")

Validity: 0.9644, Uniqueness: 0.9999, Novelty: 0.9998
